In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree
from shapely.geometry import Point, LineString
from pyproj import Transformer


kriging_file = "Kriging_GN.xlsx"
overlay_file = "Overlaid_wells.xlsx"
boundary_file = "prbbndg.shp"

target = "SSI without FI"

df = pd.read_excel(kriging_file, sheet_name="Sheet1").dropna(
    subset=["Well Number", "Easting", "Northing", "Latitude", "Longitude", target]
).copy()

X = df["Easting"].to_numpy(float)
Y = df["Northing"].to_numpy(float)
V = df[target].to_numpy(float)

coords = np.column_stack((X, Y))

gdf_wells = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["Easting"], df["Northing"]),
    crs="EPSG:32613"
)

loc_df = pd.read_excel(overlay_file, sheet_name="Producing and AP")

loc_df = loc_df.rename(columns={
    "Number": "number",
    "Latitude": "latitude",
    "Longitude": "longitude"
})

producing_wells = loc_df[loc_df["number"] <= 116].copy()
ap_wells = loc_df[loc_df["number"] >= 117].copy()

producing_gdf = gpd.GeoDataFrame(
    producing_wells,
    geometry=gpd.points_from_xy(
        producing_wells["longitude"],
        producing_wells["latitude"]
    ),
    crs="EPSG:4326"
).to_crs("EPSG:32613")

ap_gdf = gpd.GeoDataFrame(
    ap_wells,
    geometry=gpd.points_from_xy(
        ap_wells["longitude"],
        ap_wells["latitude"]
    ),
    crs="EPSG:4326"
).to_crs("EPSG:32613")


prb = gpd.read_file(boundary_file).dissolve().to_crs("EPSG:32613")

prb_buffered = gpd.GeoDataFrame(
    geometry=prb.buffer(40000),
    crs="EPSG:32613"
)

minx, miny, maxx, maxy = prb_buffered.total_bounds

buffer_x = (maxx - minx) * 0.10
buffer_y = (maxy - miny) * 0.10

minx -= buffer_x
maxx += buffer_x
miny -= buffer_y
maxy += buffer_y

grid_x = np.linspace(minx, maxx, 700)
grid_y = np.linspace(miny, maxy, 700)

grid_xx, grid_yy = np.meshgrid(grid_x, grid_y)


def idw_grid(coords_xy, values, grid_x, grid_y, k=10, p=2.0, chunk=200_000):
    tree = cKDTree(coords_xy)

    xx, yy = np.meshgrid(grid_x, grid_y)
    grid_points = np.column_stack((xx.ravel(), yy.ravel()))

    k_eff = min(k, len(values))

    z_flat = np.empty(grid_points.shape[0], dtype=float)
    dk_flat = np.empty(grid_points.shape[0], dtype=float)

    start = 0

    while start < grid_points.shape[0]:
        stop = min(start + chunk, grid_points.shape[0])

        try:
            distances, indices = tree.query(
                grid_points[start:stop],
                k=k_eff,
                workers=-1
            )
        except TypeError:
            distances, indices = tree.query(
                grid_points[start:stop],
                k=k_eff
            )

        if k_eff == 1:
            distances = distances.reshape(-1, 1)
            indices = indices.reshape(-1, 1)

        exact = distances[:, 0] == 0.0

        distances_safe = np.clip(distances, 1e-12, None)

        weights = 1.0 / (distances_safe ** p)

        if exact.any():
            weights[exact] = 0.0
            weights[exact, 0] = 1.0

        neighbor_values = values[indices]

        z_flat[start:stop] = (
            np.sum(weights * neighbor_values, axis=1)
            / np.sum(weights, axis=1)
        )

        dk_flat[start:stop] = distances[:, -1]

        start = stop

    z = z_flat.reshape(xx.shape)
    dk = dk_flat.reshape(xx.shape)

    return z, dk


idw_k = 10
idw_p = 2.0

z_idw, dk_idw = idw_grid(
    coords,
    V,
    grid_x,
    grid_y,
    k=idw_k,
    p=idw_p
)


z_min = np.nanmin(z_idw)
z_max = np.nanmax(z_idw)

z_norm = (z_idw - z_min) / (z_max - z_min)

dk_idw = np.clip(
    dk_idw,
    0,
    np.nanpercentile(dk_idw, 99.5)
)


grid_points = gpd.GeoDataFrame(
    geometry=[
        Point(x, y)
        for x, y in zip(grid_xx.ravel(), grid_yy.ravel())
    ],
    crs="EPSG:32613"
)

try:
    mask_geom = prb_buffered.union_all()
except Exception:
    mask_geom = prb_buffered.unary_union

mask = grid_points.within(mask_geom).to_numpy()

z_masked = np.full_like(z_norm, np.nan)
dk_masked = np.full_like(dk_idw, np.nan)

z_masked.ravel()[mask] = z_norm.ravel()[mask]
dk_masked.ravel()[mask] = dk_idw.ravel()[mask]


transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:32613",
    always_xy=True
)

_, y_top = transformer.transform(-107, 45.0)

top_line = LineString([
    (minx, y_top),
    (maxx, y_top)
])

top_gdf = gpd.GeoDataFrame(
    geometry=[top_line],
    crs="EPSG:32613"
)

to_latlon = Transformer.from_crs(
    "EPSG:32613",
    "EPSG:4326",
    always_xy=True
)


def set_latlon_ticks(ax):
    xticks = np.linspace(minx, maxx, 6)
    yticks = np.linspace(miny, maxy, 6)

    lon_ticks, _ = to_latlon.transform(
        xticks,
        [yticks[0]] * len(xticks)
    )

    _, lat_ticks = to_latlon.transform(
        [xticks[0]] * len(yticks),
        yticks
    )

    ax.set_xticks(xticks)
    ax.set_yticks(yticks)

    ax.set_xticklabels([f"{lon:.2f}°" for lon in lon_ticks])
    ax.set_yticklabels([f"{lat:.2f}°" for lat in lat_ticks])

    ax.set_xlabel("Longitude (°)")
    ax.set_ylabel("Latitude (°)")


def add_common_layers(ax):
    prb_buffered.boundary.plot(
        ax=ax,
        color="black",
        linewidth=1
    )

    top_gdf.plot(
        ax=ax,
        color="red",
        linestyle="--",
        linewidth=2,
        label="45°N WY–MT boundary"
    )

    gdf_wells.plot(
        ax=ax,
        color="black",
        markersize=25,
        marker="o",
        alpha=0.5,
        label="IDW wells"
    )

    producing_gdf.plot(
        ax=ax,
        color="blue",
        markersize=25,
        marker="o",
        label="Producing wells"
    )

    ap_gdf.plot(
        ax=ax,
        color="green",
        markersize=25,
        marker="^",
        label="Active permits"
    )

    ax.grid(
        True,
        linestyle="--",
        linewidth=0.5,
        alpha=0.5
    )

    set_latlon_ticks(ax)


levels = np.linspace(0, 1, 31)

fig, ax = plt.subplots(
    figsize=(8, 10),
    dpi=300
)

cf = ax.contourf(
    grid_xx,
    grid_yy,
    z_masked,
    levels=levels,
    cmap="Spectral_r"
)

add_common_layers(ax)

cbar = fig.colorbar(
    cf,
    ax=ax,
    shrink=0.8,
    aspect=20
)

cbar.set_label("Normalized Average SSI without FI")

ax.set_title(
    f"Normalized Average SSI without FI\n"
    f"IDW Interpolation"
)

ax.legend(loc="upper right", fontsize=8)

plt.tight_layout()

# plt.savefig(
#     "IDW_Normalized_SSI_without_FI_318_wells.png",
#     dpi=600,
#     bbox_inches="tight",
#     pad_inches=0.2
# )

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from scipy.spatial import cKDTree
from shapely.geometry import Point, LineString
from pyproj import Transformer

kriging_file = "Kriging_GN.xlsx"
overlay_file = "Overlaid_wells.xlsx"
boundary_file = "prbbndg.shp"

targets = ["SSI with FI", "FI"]

idw_k = 10
idw_p = 2.0

df_all = pd.read_excel(kriging_file, sheet_name="Sheet1")

loc_df = pd.read_excel(overlay_file, sheet_name="Producing and AP")
loc_df = loc_df.rename(columns={
    "Number": "number",
    "Latitude": "latitude",
    "Longitude": "longitude"
})

producing_wells = loc_df[loc_df["number"] <= 116].copy()
ap_wells = loc_df[loc_df["number"] >= 117].copy()

producing_gdf = gpd.GeoDataFrame(
    producing_wells,
    geometry=gpd.points_from_xy(
        producing_wells["longitude"],
        producing_wells["latitude"]
    ),
    crs="EPSG:4326"
).to_crs("EPSG:32613")

ap_gdf = gpd.GeoDataFrame(
    ap_wells,
    geometry=gpd.points_from_xy(
        ap_wells["longitude"],
        ap_wells["latitude"]
    ),
    crs="EPSG:4326"
).to_crs("EPSG:32613")

prb = gpd.read_file(boundary_file).dissolve().to_crs("EPSG:32613")

prb_buffered = gpd.GeoDataFrame(
    geometry=prb.buffer(40000),
    crs="EPSG:32613"
)

minx, miny, maxx, maxy = prb_buffered.total_bounds

buffer_x = (maxx - minx) * 0.10
buffer_y = (maxy - miny) * 0.10

minx -= buffer_x
maxx += buffer_x
miny -= buffer_y
maxy += buffer_y

grid_x = np.linspace(minx, maxx, 700)
grid_y = np.linspace(miny, maxy, 700)

grid_xx, grid_yy = np.meshgrid(grid_x, grid_y)

grid_points = gpd.GeoDataFrame(
    geometry=[
        Point(x, y)
        for x, y in zip(grid_xx.ravel(), grid_yy.ravel())
    ],
    crs="EPSG:32613"
)

try:
    mask_geom = prb_buffered.union_all()
except Exception:
    mask_geom = prb_buffered.unary_union

mask = grid_points.within(mask_geom).to_numpy()

transformer = Transformer.from_crs(
    "EPSG:4326",
    "EPSG:32613",
    always_xy=True
)

_, y_top = transformer.transform(-107, 45.0)

top_line = LineString([
    (minx, y_top),
    (maxx, y_top)
])

top_gdf = gpd.GeoDataFrame(
    geometry=[top_line],
    crs="EPSG:32613"
)

to_latlon = Transformer.from_crs(
    "EPSG:32613",
    "EPSG:4326",
    always_xy=True
)


def set_latlon_ticks(ax):
    xticks = np.linspace(minx, maxx, 6)
    yticks = np.linspace(miny, maxy, 6)

    lon_ticks, _ = to_latlon.transform(
        xticks,
        [yticks[0]] * len(xticks)
    )

    _, lat_ticks = to_latlon.transform(
        [xticks[0]] * len(yticks),
        yticks
    )

    ax.set_xticks(xticks)
    ax.set_yticks(yticks)

    ax.set_xticklabels([f"{lon:.2f}°" for lon in lon_ticks])
    ax.set_yticklabels([f"{lat:.2f}°" for lat in lat_ticks])

    ax.set_xlabel("Longitude (°)")
    ax.set_ylabel("Latitude (°)")


def idw_grid(
    coords_xy,
    values,
    grid_x,
    grid_y,
    k=10,
    p=2.0,
    chunk=200_000
):
    tree = cKDTree(coords_xy)

    xx, yy = np.meshgrid(grid_x, grid_y)
    grid_xy = np.column_stack((xx.ravel(), yy.ravel()))

    k_eff = min(k, len(values))

    z_flat = np.empty(grid_xy.shape[0], dtype=float)

    start = 0

    while start < grid_xy.shape[0]:
        stop = min(start + chunk, grid_xy.shape[0])

        try:
            distances, indices = tree.query(
                grid_xy[start:stop],
                k=k_eff,
                workers=-1
            )
        except TypeError:
            distances, indices = tree.query(
                grid_xy[start:stop],
                k=k_eff
            )

        if k_eff == 1:
            distances = distances.reshape(-1, 1)
            indices = indices.reshape(-1, 1)

        exact = distances[:, 0] == 0.0

        distances_safe = np.clip(
            distances,
            1e-12,
            None
        )

        weights = 1.0 / (distances_safe ** p)

        if exact.any():
            weights[exact] = 0.0
            weights[exact, 0] = 1.0

        neighbor_values = values[indices]

        z_flat[start:stop] = (
            np.sum(weights * neighbor_values, axis=1)
            / np.sum(weights, axis=1)
        )

        start = stop

    return z_flat.reshape(xx.shape)


def add_common_layers(ax, gdf_wells):
    prb_buffered.boundary.plot(
        ax=ax,
        color="black",
        linewidth=1
    )

    top_gdf.plot(
        ax=ax,
        color="red",
        linestyle="--",
        linewidth=2,
        label="45°N WY–MT boundary"
    )

    gdf_wells.plot(
        ax=ax,
        color="black",
        markersize=22,
        marker="o",
        alpha=0.5,
        label="IDW wells"
    )

    producing_gdf.plot(
        ax=ax,
        color="blue",
        markersize=25,
        marker="o",
        label="Producing wells"
    )

    ap_gdf.plot(
        ax=ax,
        color="green",
        markersize=25,
        marker="^",
        label="Active permits"
    )

    ax.grid(
        True,
        linestyle="--",
        linewidth=0.5,
        alpha=0.5
    )

    set_latlon_ticks(ax)


for target in targets:
    df = df_all.dropna(
        subset=[
            "Well Number",
            "Easting",
            "Northing",
            "Latitude",
            "Longitude",
            target
        ]
    ).copy()

    X = df["Easting"].to_numpy(float)
    Y = df["Northing"].to_numpy(float)
    V = df[target].to_numpy(float)

    coords = np.column_stack((X, Y))

    gdf_wells = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(
            df["Easting"],
            df["Northing"]
        ),
        crs="EPSG:32613"
    )

    z_idw = idw_grid(
        coords,
        V,
        grid_x,
        grid_y,
        k=idw_k,
        p=idw_p
    )

    print(f"\n{target}")

    z_min = np.nanmin(z_idw)
    z_max = np.nanmax(z_idw)

    z_norm = (z_idw - z_min) / (z_max - z_min)

    z_masked = np.full_like(z_norm, np.nan)
    z_masked.ravel()[mask] = z_norm.ravel()[mask]

    levels = np.linspace(0, 1, 31)

    fig, ax = plt.subplots(
        figsize=(8, 10),
        dpi=300
    )

    cf = ax.contourf(
        grid_xx,
        grid_yy,
        z_masked,
        levels=levels,
        cmap="Spectral_r"
    )

    add_common_layers(ax, gdf_wells)

    cbar = fig.colorbar(
        cf,
        ax=ax,
        shrink=0.8,
        aspect=20
    )

    cbar.set_label(
        f"Normalized Average {target}"
    )

    ax.set_title(
        f"Normalized Average {target}\n"
        f"IDW Interpolation"
    )

    ax.legend(
        loc="upper right",
        fontsize=8
    )

    plt.tight_layout()

    safe_name = target.replace(" ", "_")

    # plt.savefig(
    #     f"IDW_Normalized_{safe_name}_318_wells_k{idw_k}_p{idw_p}.png",
    #     dpi=600,
    #     bbox_inches="tight",
    #     pad_inches=0.2
    # )

    plt.show()